<a href="https://colab.research.google.com/github/neerajvr51-hub/Medical-Insurance-Analytics/blob/main/healthcare_fraud_det.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***1.Data Loading & Inspection***

In [1]:
import numpy as np
import pandas as pd

In [2]:
#To get a complete view of the columns
pd.set_option('display.max_columns', None)

In [3]:
#Loading the dataset
df = pd.read_csv("/content/healthcare_fraud_detection.csv")

In [4]:
#Basic understanding of data
df.head(10)

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0
5,P0241,C0000005,44,Female,E11.9,80053,220.99,174.05,Medicaid,2024-12-20,26,74,Orthopedics,CA,Approved,0,2,Emergency,0,1.0
6,P0196,C0000006,50,Male,E78.5,93000,458.60,372.31,Medicare,2022-02-20,2,84,Cardiology,OH,Approved,0,0,Outpatient,1,0.0
7,P0071,C0000007,1,Male,I10,85025,934.34,321.60,NaN,2022-07-13,6,114,NaN,PA,Pending,1,2,Emergency,0,NaN
8,P0097,C0000008,34,Female,M54.5,97110,478.11,412.48,Medicare,2021-05-25,24,65,Pulmonology,IL,Pending,0,5,Inpatient,1,4.0
9,P0125,C0000009,47,Male,K21.9,99213,426.16,335.80,Self-Pay,2022-05-29,5,56,Internal Medicine,TX,Approved,0,2,Emergency,0,4.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         9650 non-null   object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

# *2.Data Cleaning & Handling Missing Value*

In [6]:
#How much percentage of 'null values' each column have
df.isna().mean() * 100

,0
Provider_ID,0.0
Claim_ID,0.0
Patient_Age,0.0
Patient_Gender,0.0
Diagnosis_Code,0.0
Procedure_Code,0.0
Claim_Amount,0.0
Approved_Amount,0.0
Insurance_Type,3.5
Claim_Submission_Date,0.0


In [7]:
#Replacing missing categorical data with 'Unknown'. I could have dropped these rows or imputed them with the mode, but healthcare data is highly sensitive. These missing values could correlate with fraudulent claims, so I retained them to avoid data loss.
df['Insurance_Type'] = df['Insurance_Type'].fillna('Unknown')
df['Provider_Specialty'] = df['Provider_Specialty'].fillna('Unknown')

In [8]:
#Filling this with median because visits are in count
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(df['Prior_Visits_12m'].median())

In [9]:
#All null values are replaced
df.isna().sum()

,0
Provider_ID,0
Claim_ID,0
Patient_Age,0
Patient_Gender,0
Diagnosis_Code,0
Procedure_Code,0
Claim_Amount,0
Approved_Amount,0
Insurance_Type,0
Claim_Submission_Date,0


In [10]:
#Checking duplicates
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
9995,False
9996,False
9997,False
9998,False


In [11]:
#There are no duplicate records in this dataset.

# *3.Exploratory Data Analysis & Anomaly Detection*

In [12]:
#Converting column "Claim_Submission_Date" into Datetime format
df['Claim_Submission_Date'] = pd.to_datetime(df['Claim_Submission_Date'])

In [13]:
#Extracting temporal features (Year, Month, Day) to analyze fraudulent behavior trends,
#such as spikes before major holidays or end-of-quarter.
df['Submission_Year'] = df['Claim_Submission_Date'].dt.year
df['Submission_Month'] = df['Claim_Submission_Date'].dt.month
df['Submission_Month_Name'] = df['Claim_Submission_Date'].dt.strftime('%B')
df['Submission_Day_of_Week'] = df['Claim_Submission_Date'].dt.strftime('%A')

In [14]:
#Subtracts the Approved_Amount from the Claim_Amount to know the the overbilling margin
df['Amount_Gap'] = df['Claim_Amount']-df['Approved_Amount']
df['Amount_Gap']


,Amount_Gap
0,50.35
1,6.17
2,61.63
3,46.04
4,118.05
...,...
9995,385.19
9996,27.84
9997,57.30
9998,57.07


In [15]:
df.head()

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m,Submission_Year,Submission_Month,Submission_Month_Name,Submission_Day_of_Week,Amount_Gap
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0,2024,9,September,Sunday,50.35
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0,2022,9,September,Monday,6.17
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0,2022,4,April,Monday,61.63
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0,2023,10,October,Wednesday,46.04
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0,2023,9,September,Tuesday,118.05


In [16]:
df.drop('Submission_Month',axis=1)

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m,Submission_Year,Submission_Month_Name,Submission_Day_of_Week,Amount_Gap
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0,2024,September,Sunday,50.35
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0,2022,September,Monday,6.17
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0,2022,April,Monday,61.63
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0,2023,October,Wednesday,46.04
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0,2023,September,Tuesday,118.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,P0289,C0009995,79,Male,E78.5,71046,590.19,205.00,Private,2024-09-24,2,115,Internal Medicine,PA,Approved,1,2,Inpatient,1,3.0,2024,September,Tuesday,385.19
9996,P0248,C0009996,66,Male,K21.9,93000,219.73,191.89,Self-Pay,2023-01-04,6,73,Orthopedics,IL,Approved,0,5,Inpatient,0,6.0,2023,January,Wednesday,27.84
9997,P0122,C0009997,43,Female,F41.9,99213,505.24,447.94,Medicare,2024-06-30,6,59,Neurology,OH,Rejected,0,5,Emergency,1,0.0,2024,June,Sunday,57.30
9998,P0072,C0009998,39,Female,I10,93000,564.25,507.18,Medicare,2022-03-15,8,55,Cardiology,TX,Pending,0,5,Emergency,0,6.0,2022,March,Tuesday,57.07


In [17]:
# Calculate the percentage of each class in the Is_Fraud column to know the percentage of fraud claims
df['Is_Fraud'].value_counts(normalize=True) * 100

,proportion
Is_Fraud,
0,91.71
1,8.29


In [18]:
#Checking how much money is at risk due to fraud claims
df.groupby('Is_Fraud').agg({'Claim_Amount':['sum','mean','median']})

Claim_Amount                    
                  sum        mean  median
Is_Fraud                                 
0          4906561.60  535.008352  442.88
1           821482.46  990.931797  831.16

In [19]:
#Checking Average of Amount gap between billed amount and approved amount
df.groupby('Is_Fraud').agg({'Amount_Gap':['mean','median']})

Amount_Gap        
                mean  median
Is_Fraud                    
0          65.854094   48.33
1         445.059819  355.82

In [20]:
#While doing my analysis. I noticed that rejected claims also have approved amount, hence checked how many rejected claims have approved amount
rejected_with_money = df[(df['Claim_Status'] == 'Rejected') & (df['Approved_Amount'] > 0)]
rejected_with_money

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m,Submission_Year,Submission_Month,Submission_Month_Name,Submission_Day_of_Week,Amount_Gap
16,P0100,C0000016,48,Female,K21.9,99213,559.84,376.77,Private,2023-09-30,2,81,Pulmonology,PA,Rejected,1,2,Outpatient,1,4.0,2023,9,September,Saturday,183.07
28,P0292,C0000028,20,Male,E78.5,36415,886.53,328.02,Medicaid,2021-09-09,5,71,Internal Medicine,PA,Rejected,1,5,Outpatient,0,5.0,2021,9,September,Thursday,558.51
33,P0003,C0000033,68,Female,I25.10,99214,182.85,138.25,Medicare,2022-03-12,28,76,Pulmonology,GA,Rejected,0,2,Emergency,1,4.0,2022,3,March,Saturday,44.60
38,P0039,C0000038,66,Male,F41.9,99213,455.56,403.60,Private,2022-05-13,7,65,Internal Medicine,IL,Rejected,0,3,Outpatient,0,2.0,2022,5,May,Friday,51.96
39,P0169,C0000039,33,Male,I25.10,36415,636.91,478.50,Private,2022-10-10,8,56,Internal Medicine,NY,Rejected,0,5,Outpatient,0,1.0,2022,10,October,Monday,158.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9970,P0240,C0009970,51,Male,M54.5,97110,257.01,246.07,Medicare,2023-09-01,25,69,Neurology,IL,Rejected,0,3,Outpatient,0,1.0,2023,9,September,Friday,10.94
9976,P0002,C0009976,37,Female,I25.10,87086,482.14,373.93,Medicare,2022-04-11,14,119,Orthopedics,PA,Rejected,0,0,Outpatient,1,4.0,2022,4,April,Monday,108.21
9981,P0082,C0009981,60,Male,J06.9,71046,833.17,425.09,Private,2021-05-11,2,54,Internal Medicine,NY,Rejected,1,1,Emergency,0,1.0,2021,5,May,Tuesday,408.08
9988,P0074,C0009988,53,Male,J18.9,71046,641.57,498.90,Private,2023-04-08,5,62,Neurology,TX,Rejected,0,5,Inpatient,0,2.0,2023,4,April,Saturday,142.67


In [21]:
#Checking number of rejected but approved claims
print("Number of anomaly rows:", len(rejected_with_money))

Number of anomaly rows: 1748


In [22]:
rejected_with_money['Is_Fraud'].value_counts(normalize=True) * 100

,proportion
Is_Fraud,
0,76.887872
1,23.112128


In [23]:
#Checked the specialty wise fraud claim percentage
df.groupby('Provider_Specialty')['Is_Fraud'].mean().sort_values(ascending=False)*100

,Is_Fraud
Provider_Specialty,
General Practice,9.691161
Internal Medicine,8.667657
Unknown,8.571429
Orthopedics,8.539604
Pulmonology,7.808765
Cardiology,7.320644
Neurology,6.992084


In [24]:
#Checking providers with most fraud billes
df.groupby('Provider_ID').agg({'Is_Fraud':['count','mean']}).sort_values(by=('Is_Fraud','mean'), ascending=False)

Is_Fraud          
               count      mean
Provider_ID                   
P0086             38  0.421053
P0159             20  0.350000
P0110             45  0.333333
P0104             34  0.323529
P0223             44  0.318182
...              ...       ...
P0053             22  0.000000
P0031             23  0.000000
P0024             38  0.000000
P0273             30  0.000000
P0294             33  0.000000

[300 rows x 2 columns]

In [25]:
#Checking if fraud providers targeting a specific age group for fraud claims?
df.groupby('Is_Fraud').agg({'Patient_Age':['mean','median','min','max']})

Patient_Age               
                mean median min max
Is_Fraud                           
0          49.846036   50.0   1  95
1          48.747889   48.0   1  95

In [26]:
#Checking if providers targeting specific day for fraud claims like weekends
df.groupby('Submission_Day_of_Week')['Is_Fraud'].mean().sort_values(ascending=False)*100

,Is_Fraud
Submission_Day_of_Week,
Friday,9.097473
Sunday,8.692722
Wednesday,8.510638
Tuesday,8.396947
Monday,7.918711
Saturday,7.719054
Thursday,7.703180


In [27]:
# 1. Separate target variable and features
y = df['Is_Fraud']

# Drop the target, unique IDs, and raw text dates
X = df.drop(columns=['Is_Fraud', 'Claim_ID', 'Claim_Submission_Date'])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (10000, 22)
Target shape: (10000,)


In [28]:
df.to_csv('healthcare_fraud_cleaned.csv', index=False)